<a href="https://colab.research.google.com/github/ClassNeuralNetwork/classification-fatigue-detection/blob/main/Deteccao_Sonolencia_MobileNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import os
import cv2
import random
from tqdm.notebook import tqdm
from numpy.random import shuffle
import numpy as np
import pandas as pd
from tensorflow.keras.layers import Input, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("serenaraju/yawn-eye-dataset-new")

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

DATASET = os.path.join(path, "dataset_new")
TRAIN_DIR = os.path.join(DATASET, "train")
TEST_DIR = os.path.join(DATASET, "test")

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASS = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASS}): {CLASS_NAMES}")

In [ ]:
def load_image(directory):
    images, labels = [], []
    for category in os.listdir(directory):
        category_path = os.path.join(directory, category)
        if not os.path.isdir(category_path):
            continue
        for filename in tqdm(os.listdir(category_path), desc=category):
            img = cv2.imread(os.path.join(category_path, filename))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMAGE_SIZE)
            img = preprocess_input(img)
            images.append(img)
            labels.append(category)
    return np.array(images, dtype='float32'), labels

print("Carregando treino...")
train_images, train_labels = load_image(TRAIN_DIR)
print("Carregando teste...")
test_images, test_labels = load_image(TEST_DIR)

In [ ]:
binary_labels = []
for label in train_labels:
    if label in ['Closed', 'yawn']:
        binary_labels.append(1) #fadiga
    else:
        binary_labels.append(0) #não fadiga

y_train_all = np.array(binary_labels)

test_binary_labels = []
for label in test_labels:
    if label in ['Closed', 'yawn']:
        test_binary_labels.append(1) #fadiga
    else:
        test_binary_labels.append(0) #não fadiga

y_test = np.array(test_binary_labels)

In [ ]:
classes = np.unique(y_train_all)

class_weights_array = class_weight.compute_class_weight(
    'balanced',
    classes=classes,
    y=y_train_all
    )

class_weight_dict = dict(enumerate(class_weights_array))

print(class_weight_dict)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_images, y_train_all,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_all,
    )

print(f'\nX_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {test_images.shape}')

In [ ]:
def im(img):
  img = (img + 1.0) / 2.0
  return np.clip(img, 0, 1)

plt.figure(figsize = (10, 10))
for i in range(min(25, len(X_train))):
    ax = plt.subplot(5, 5, i + 1)
    plt.imshow(im(X_train[i]))
    plt.title(CLASS_NAMES[y_train[i]])
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.axis("off")
plt.suptitle("Conjunto de amostras", fontsize=15, color='red')
plt.tight_layout()
plt.show()

In [ ]:
argumentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1),
])

In [ ]:
tf.keras.backend.clear_session()

base_model = MobileNet(
  input_shape=(*IMAGE_SIZE, 3),
  include_top=False,
  alpha=0.5,
  weights="imagenet",
  pooling='avg',
  name="Sonolencia",
)

base_model.trainable = False

inputs = Input(shape=(*IMAGE_SIZE, 3))
x = argumentation(inputs)
x = base_model(x, training=False)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(1, activation='sigmoid')(x)
model_transfer = Model(inputs=inputs, outputs=predictions)
model_transfer.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

In [ ]:
model_transfer.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model_transfer.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
)

In [ ]:
model_transfer.save('model_sonolencia.h5')
model_transfer.save('model_sonolencia.keras')

In [ ]:
plt.figure(figsize=(7, 3))
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], 'r--', label='accuracy')
plt.plot(history.history['val_accuracy'], 'm', label='val_accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.figure(figsize=(7, 3))
plt.subplot(1,2,2)
plt.plot(history.history['loss'], 'r--', label='loss')
plt.plot(history.history['val_loss'], 'm', label='val_loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model_transfer.predict(test_images)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()
y_true = y_test

print(classification_report(
    y_true, y_pred,
    target_names=["Não Fadiga (0)", "Fadiga (1)"]
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Não Fadiga", "Fadiga"]
)

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Matriz de confusão")
plt.tight_layout()
plt.show()